# In-Context Learning for Address Extraction

Evaluate Few-Shot In-Context Learning on the same model used for fine-tuning (Llama-3.2-1B).
Compare performance with different numbers of in-context examples.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import torch
import pandas as pd
from typing import List, Dict
from tqdm import tqdm
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from data_preparation.preprocess_data import preprocess_data, make_splits, evaluate_predictions


/home/zuza/zuza/studia/NLP-address-data-parsing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#check cuda
!nvidia-smi

Sat May  2 12:51:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2060        Off |   00000000:26:00.0  On |                  N/A |
|  0%   50C    P3             37W /  170W |    1295MiB /   6144MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
torch.cuda.is_available()

True

In [7]:
torch.cuda.empty_cache()

## Load Model & Data

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/Llama-3.2-1B-Instruct"
LOAD_IN_4BIT    = True
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Your task is to extract address components from the user’s message. Ignore any missing information and exclude those fields from your output. Return the data in JSON format using this structure: {'house_number', 'street', 'city', 'country', 'postal_code', 'state', 'country'}. Return ONLY VALID JSON output, nothing else."
)

# SYSTEM_PROMPT = (
#     "You are a helpful assistant who gets messages containing addresses from different countries. Your task is to extract address components from the user's messages. "
#     "You always retun JSON in exavtly same format as shown below, alawys include all fields in the same order that are present in the input. If some are missing, include them as empty strings. "
#     "{'house_number': '<house number>', 'street': '<street name>', 'city': '<city name>', 'postal_code': '<postal or zip code>', 'state': '<state or province name>', 'country': '<country name>' }"
#     "If you find some missspelled words, correct them. If you are not sure about some component, do not include it in the output."
#     "One component can be part of another one, for example 'street' can include 'house_number' or 'postal_code' can be part of 'street'. In such cases, include only the most specific component."
#     "Use exactly these keys: house_number, street, city, postal_code, state, country.\n"
#     "Example output for message 'Warsaw, Masovia, Poland': {'house_number': '',  'street': '', 'city': 'Warsaw', 'postal_code': '', 'state': '', 'country': 'Poland'}"
#     "Example output for message 'ul. Marszałkowska 10, 00-001 Warszawa': {'house_number': '',  'street': 'ul. Marszałkowska', 'city': 'Warszawa', 'postal_code': '00-001', 'state': '', 'country': ''}" 
#     "Example output for message 'Ep Epovska 81 Lubihjë XK Epërme Epërme': {'house_number': '81',  'street': 'Epovska', 'city': 'Lubinjë E Epërme', 'country': 'XK', 'postal_code': '','state': ''}"
#     "Example output for message 'Studençan 18 Killosa XK': {'house_number': '18',  'street': 'Studençan', 'city': 'Killosa', 'country': 'XK', 'postal_code': '','state': ''}"
#     "Example output for message '8501 Gullegemsestsenweg VLG BE 158 kKortrijk': {'house_number': '158',  'street': 'Gullegemsestsenweg', 'city': 'Kortrijk', 'country': 'BE', 'postal_code': '8501', 'state': 'VLG'}"
#     "Example output for message '尾道市 高須町 1590-3 広島県 J;: {'house_number': '1590-3',  'street': '高須町', 'city': '尾道市', 'state': '広島県', 'country': 'JP', 'postal_code': '', state': '広島県'}"
#     "Example output for message '63376 MO Peters St Ct Carson 37 IS mailer': {'house_number': '37', 'street': 'Carson Ct', 'city': 'St Peters', 'country': 'US', 'postal_code': '63376', 'state': 'MO'}"
#     "Example output for message 'Mariendorf 12107 Anhkogelweg postal A EE 80.0 code': {'house_number': '80.0 A', 'street': 'Ankogelweg', 'city': 'Mariendorf', 'country': 'DE', 'postal_code': '12107', 'state': ''}"
#     "Example output for message '1315/- rVia Kontalbano Santardangdlo Di Tomzgna 45 IT': {'house_number': '1315/-', 'street': 'Via Kontalbano', 'city': 'Santardangdlo Di Romagna', 'country': 'IT', 'postal_code': '', 'state': '45'}"
#     "Example output for message 'd1190f-6 d1190f-6 岩出 度会郡玉城町 三重県 JP': {'house_number': '1190-6', 'street': '岩出', 'city': '度会郡玉城町', 'country': 'JP', 'postal_code': '', 'state': '三重県'}"
#     "Example output for message '2 Rue Des Fossés Apt 12 Tournai WAL 7500 BE952': {'house_number': '2', 'street': 'Rue Des Fossés Apt 12', 'city': 'Tournai', 'country': 'BE', 'postal_code': '7500', 'state': 'WAL'}"

# )


# ── Evaluation parameters ─────────────────────────────────────────────────────
NUM_SHOTS_LIST = [0, 2, 4, 8]  # Test different numbers of few-shot examples
BATCH_SIZE = 8
SEED = 42
print("DEVICE:", DEVICE)

DEVICE: cuda


## In-Context Learning Functions

In [6]:
def load_model_and_tokenizer():
    """Load base model (not fine-tuned) for in-context learning."""
    bnb_config = BitsAndBytesConfig(load_in_4bit=LOAD_IN_4BIT)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    # Left-padding is required for correct batch generation with decoder-only models
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    model.eval()
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()
print(f"Model loaded: {MODEL_NAME}")

# Load data
df = preprocess_data()
print(f"Total data: {len(df)} examples")


Loading weights: 100%|██████████| 146/146 [00:00<00:00, 273.18it/s]


Model loaded: unsloth/Llama-3.2-1B-Instruct
Loading dataset...
Cleaning data...
Total data: 819849 examples


In [8]:
df.loc[46].to_dict()

{'input': 'Eldhani XK Bellanicë Ahi, 26',
 'target': {'house_number': '26',
  'street': 'Agim Elshani',
  'city': 'Bellanicë',
  'country': 'XK',
  'postal_code': '',
  'state': ''}}

## Evaluate In-Context Learning on K-Fold Splits

In [9]:
def create_few_shot_prompt(examples: List[Dict], user_input: str) -> List[Dict]:
    """Create chat messages with in-context examples."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for example in examples:
        target = {k: v for k, v in example["target"].items() if v}
        messages.append({"role": "user", "content": example["input"]})
        messages.append({"role": "assistant", "content": json.dumps(target, ensure_ascii=False)})

    messages.append({"role": "user", "content": user_input})
    return messages


def batch_generate_with_icl(
    texts: List[str],
    few_shot_examples: List[Dict],
    model,
    tokenizer,
    batch_size: int = 16,
) -> List[str]:
    torch.cuda.empty_cache()
    preds = []
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch = texts[i : i + batch_size]

        input_ids_list = []
        attention_list = []

        for text in batch:
            messages = create_few_shot_prompt(few_shot_examples, text)
            inp = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            input_ids_list.append(inp["input_ids"].squeeze(0))
            attention_list.append(inp["attention_mask"].squeeze(0))

        # Left-pad so all sequences are aligned at the right (generation starts correctly)
        max_len = max(ids.shape[0] for ids in input_ids_list)
        n = len(input_ids_list)
        input_ids_padded = torch.full((n, max_len), pad_id, dtype=torch.long)
        attention_padded = torch.zeros(n, max_len, dtype=torch.long)

        for j, (ids, att) in enumerate(zip(input_ids_list, attention_list)):
            seq_len = ids.shape[0]
            input_ids_padded[j, max_len - seq_len :] = ids
            attention_padded[j, max_len - seq_len :] = att

        input_ids_padded = input_ids_padded.to(DEVICE)
        attention_padded = attention_padded.to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids_padded,
                attention_mask=attention_padded,
                max_new_tokens=128,
                do_sample=False,
            )

        # Generated tokens start after max_len (the padded input length)
        for out in outputs:
            gen = out[max_len:]
            preds.append(tokenizer.decode(gen, skip_special_tokens=True))

    return preds


## Results Summary

In [10]:
icl_results = {}

for num_shots in NUM_SHOTS_LIST:
    print(f"\n{'='*70}")
    print(f" EVALUATING IN-CONTEXT LEARNING WITH {num_shots}-SHOT EXAMPLES")
    print(f"{'='*70}")

    fold_metrics = []

    for fold_idx, split in enumerate(make_splits(df)):
        print(f"\n  Fold {fold_idx + 1}")

        train_df = split["train"]
        test_df = split["test"]

        if num_shots > 0:
            few_shot_examples = train_df.sample(
                n=min(num_shots, len(train_df)), random_state=SEED
            ).to_dict("records")
        else:
            few_shot_examples = []

        texts = test_df["input"].astype(str).tolist()
        preds = batch_generate_with_icl(texts, few_shot_examples, model, tokenizer, batch_size=BATCH_SIZE)

        golds = test_df["target"].tolist()
        metrics = evaluate_predictions(golds, preds)

        print(f"    examples:              {metrics['counts']['examples']}")
        print(f"    exact_match:           {metrics['exact_match']:.4f}")
        print(f"    overall_item_accuracy: {metrics['overall_item_accuracy']:.4f}")
        print("    per_field_accuracy:")
        for field, acc in metrics["per_field_accuracy"].items():
            print(f"      {field:<15} {acc:.4f}")

        fold_metrics.append(metrics)

        # Only evaluate first fold for speed
        break

    icl_results[num_shots] = fold_metrics[0]



 EVALUATING IN-CONTEXT LEARNING WITH 0-SHOT EXAMPLES
Downsampling data from 819849 to 10000 samples...

  Fold 1


Generating:   0%|          | 0/188 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Generating: 100%|██████████| 188/188 [1:22:28<00:00, 26.32s/it]


    examples:              1500
    exact_match:           0.0000
    overall_item_accuracy: 0.0509
    per_field_accuracy:
      house_number    0.0213
      street          0.0000
      city            0.0000
      country         0.0007
      postal_code     0.2460
      state           0.0373

 EVALUATING IN-CONTEXT LEARNING WITH 2-SHOT EXAMPLES
Downsampling data from 819849 to 10000 samples...

  Fold 1


Generating:   0%|          | 0/188 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 324.00 MiB. GPU 0 has a total capacity of 5.58 GiB of which 172.88 MiB is free. Including non-PyTorch memory, this process has 4.53 GiB memory in use. Of the allocated memory 4.10 GiB is allocated by PyTorch, and 338.16 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
fields = ["house_number", "street", "city", "country", "postal_code", "state"]

rows = []
for num_shots, metrics in icl_results.items():
    row = {
        "num_shots": num_shots,
        "exact_match": round(metrics["exact_match"], 4),
        "overall_item_acc": round(metrics["overall_item_accuracy"], 4),
    }
    for f in fields:
        row[f] = round(metrics["per_field_accuracy"].get(f, 0.0), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("IN-CONTEXT LEARNING RESULTS SUMMARY")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nNote: Results from base model (not fine-tuned). Fold 1 only.")
